## SETUP & KONFIGURASI

In [ ]:
pip install clickhouse-connect==0.15.1 pandas==3.0.2 numpy==2.1.3 scikit-learn==1.8.0 matplotlib==3.9.3 seaborn==0.13.2 scipy==1.17.1

In [ ]:
import clickhouse_connect
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import scipy

# Tampilkan versi semua library (WAJIB)
print("=== VERSI LIBRARY ===")
print(f"clickhouse-connect : {clickhouse_connect.__version__}")
print(f"pandas             : {pd.__version__}")
print(f"numpy              : {np.__version__}")
print(f"scikit-learn       : {sklearn.__version__}")
print(f"matplotlib         : {matplotlib.__version__}")
print(f"seaborn            : {sns.__version__}")
print(f"scipy              : {scipy.__version__}")

# Variabel konfigurasi koneksi
HOST      = 'overproficiently-unbenevolent-gregoria.ngrok-free.dev'
PORT      = 443
USERNAME  = 'mahasiswa'
PASSWORD  = 'bigdata123'
DATABASE  = 'praktikum'
TABLE_SRC = 'final_pipeline'
TABLE_DST = 'clustering_results'
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
print(f"\nRandom seed ditetapkan: {RANDOM_SEED}")

# Koneksi ke ClickHouse
print("\nMenghubungkan ke ClickHouse...")
client = clickhouse_connect.get_client(
    host=HOST,
    port=PORT,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE,
    secure=True
)
print(f"Koneksi berhasil! Versi server: {client.server_version}")

## EXPORT DATA HASIL PREPROCESSING KE PANDAS

In [ ]:
# Query agregasi per perusahaan — fitur statistik pola publikasi
# Keputusan: clustering dilakukan pada level perusahaan (361 entitas unik)
# Alasan: kolom teks tidak bisa langsung di-cluster tanpa NLP;
#         fitur waktu (hour, minute, day_of_week, is_weekend) merepresentasikan
#         pola perilaku publikasi berita setiap perusahaan.

print("Mengambil dan mengagregasi data dari ClickHouse (dengan Cyclical Transform)...")

query = """
    SELECT
        cik_clean,
        cleaned_company,

        -- Volume publikasi asli (nanti dilogaritma di Python)
        count(*) AS total_articles,

        -- Transformasi Sirkular untuk Jam (0-23)
        avg(sin(hour * 2 * pi() / 24)) AS avg_sin_hour,
        avg(cos(hour * 2 * pi() / 24)) AS avg_cos_hour,
        stddevPop(hour) AS std_hour,

        -- Pola menit dibiarkan linear karena 0 dan 59 jarang punya signifikansi siklus di publikasi artikel
        avg(minute) AS avg_minute,
        stddevPop(minute) AS std_minute,

        -- Transformasi Sirkular untuk Hari (1-7)
        avg(sin(day_of_week * 2 * pi() / 7)) AS avg_sin_day_of_week,
        avg(cos(day_of_week * 2 * pi() / 7)) AS avg_cos_day_of_week,
        stddevPop(day_of_week) AS std_day_of_week,

        -- Proporsi publikasi di akhir pekan
        avg(is_weekend) AS ratio_weekend

    FROM praktikum.final_pipeline
    GROUP BY cik_clean, cleaned_company
    ORDER BY total_articles DESC
"""

df = client.query_df(query)

print(f"✅ Data berhasil diambil!")
print(f"   Jumlah perusahaan (baris) : {df.shape[0]}")
print(f"   Jumlah fitur (kolom)      : {df.shape[1]}")
print(f"   Ukuran DataFrame          : {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"\nKolom: {list(df.columns)}")
print(f"\nSample data:")
print(df.head())
print(f"\nStatistik deskriptif:")
print(df.describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. EKSPLORASI (EDA)
print("=== STATISTIK DESKRIPTIF ===")
print(df.describe().round(4))

# Cek Korelasi (Bisa di-uncomment jika ingin melihat visualisasinya)
# plt.figure(figsize=(10, 8))
# sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
# plt.title('Korelasi Antar Fitur (Sirkular)')
# plt.show()

# 2. FEATURE ENGINEERING
# Transformasi skewness untuk volume artikel
df['total_articles_log'] = np.log1p(df['total_articles'])

# Tidak perlu df.drop() karena kolom redundan sudah tidak ditarik dari SQL
df_model = df.copy()

# 3. PENYIAPAN FITUR & IMPUTASI
feature_cols = [
    'total_articles_log',
    'avg_sin_hour',
    'avg_cos_hour',
    'std_hour',
    'avg_minute',
    'std_minute',
    'avg_sin_day_of_week',
    'avg_cos_day_of_week',
    'std_day_of_week',
    'ratio_weekend'
]

df_identity = df_model[['cik_clean', 'cleaned_company']].copy()
X = df_model[feature_cols].copy()

# Isi NaN pada std (untuk perusahaan dengan 1 artikel)
X = X.fillna(0)

# 4. SCALING
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("\n=== HASIL SCALING (StandardScaler) ===")
print("Statistik setelah scaling (mean ≈ 0, std ≈ 1):")
print(X_scaled_df.describe().round(4))

## PRE-PROCESSING DI PYTHON

### 1. Pilih fitur untuk clustering

In [ ]:
# Keputusan: kolom cik_clean & cleaned_company digunakan sebagai
# identitas, BUKAN sebagai fitur model (teks tidak di-encode
# karena tidak merepresentasikan pola perilaku secara numerik)

# 1. PERUBAHAN: Daftar fitur diperbarui (memakai log, membuang yang redundan)
feature_cols = [
    'total_articles_log',
    'avg_sin_hour',
    'avg_cos_hour',
    'std_hour',
    'avg_minute',
    'std_minute',
    'avg_sin_day_of_week',
    'avg_cos_day_of_week',
    'std_day_of_week',
    'ratio_weekend'
]

# 2. Mengambil data dari df_model
df_identity = df_model[['cik_clean', 'cleaned_company']].copy()

# Matrix fitur
X = df_model[feature_cols].copy()

print("=== 4.1 FITUR YANG DIGUNAKAN ===")
print(f"Jumlah fitur  : {len(feature_cols)}")
print(f"Jumlah sampel : {X.shape[0]}")
print(f"\nCek missing values setelah agregasi:")
print(X.isnull().sum())

# Isi NaN pada std (perusahaan dengan 1 artikel → std = NaN → 0)
X = X.fillna(0)
print(f"\nMissing values setelah fillna(0): {X.isnull().sum().sum()}")

### 2 - Scaling dengan StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

# Keputusan: StandardScaler dipakai karena clustering berbasis
# jarak (K-Means, DBSCAN) sensitif terhadap skala fitur.
# total_articles sudah ditransformasi (log) sehingga distribusinya lebih baik.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X) # X di sini sudah berisi fitur final yang bersih
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("\n=== 4.2 HASIL SCALING (StandardScaler) ===")
print("Statistik setelah scaling (mean ≈ 0, std ≈ 1):")
print(X_scaled_df.describe().round(4))

### 3 - PCA untuk reduksi dimensi (visualisasi 2D)

In [ ]:

# Keputusan: PCA 2 komponen digunakan KHUSUS untuk visualisasi
# scatter plot oleh Orang 4. Model clustering tetap memakai
# X_scaled (13 fitur) agar informasi tidak hilang.

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_scaled)

print("\n=== 4.3 HASIL PCA ===")
print(f"Explained variance ratio : {pca.explained_variance_ratio_}")
print(f"Total variance explained : {sum(pca.explained_variance_ratio_):.2%}")
print(f"Shape X_pca              : {X_pca.shape}")

# Simpan PCA ke DataFrame dengan identitas
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca = pd.concat([df_identity.reset_index(drop=True), df_pca], axis=1)

print(f"\nSample df_pca:")
print(df_pca.head())

### 4 - Ringkasan variabel untuk Orang 2

In [ ]:
print("\n=== RINGKASAN OUTPUT UNTUK ORANG 2 ===")
print(f"X_scaled  → shape {X_scaled.shape}  | input model clustering")
print(f"X_pca     → shape {X_pca.shape}     | visualisasi scatter 2D")
print(f"df        → shape {df.shape}        | DataFrame lengkap")
print(f"df_pca    → shape {df_pca.shape}    | PCA + identitas perusahaan")
print(f"RANDOM_SEED = {RANDOM_SEED}")
print("\n✅ Pre-processing selesai! Data siap untuk modeling (Orang 2).")

### PEMODELAN ML (CLUSTERING)

#### K-MEANS

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("Mencari nilai K optimal...")

# Menyimpan riwayat metrik untuk setiap pengujian K
inertia_values = []
silhouette_scores = []

# Kita akan menguji jumlah klaster dari 2 hingga 10
k_range = range(2, 11)

for k in k_range:
    # 1. Latih model dengan jumlah klaster K
    kmeans_test = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto")
    labels_test = kmeans_test.fit_predict(X_scaled)
    
    # 2. Simpan nilai Inertia (untuk Metode Elbow)
    inertia_values.append(kmeans_test.inertia_)
    
    # 3. Simpan nilai Silhouette
    sil_score = silhouette_score(X_scaled, labels_test)
    silhouette_scores.append(sil_score)

# VISUALISASI GRAFIK EVALUASI
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Metode Elbow (Inertia)
ax1.plot(k_range, inertia_values, marker='o', linestyle='-', color='b')
ax1.set_title('Metode Elbow (Cari Titik Siku/Patahan)')
ax1.set_xlabel('Jumlah Klaster (k)')
ax1.set_ylabel('Inertia / WCSS (Semakin kecil semakin baik)')
ax1.set_xticks(k_range)
ax1.grid(True, linestyle='--', alpha=0.7)

# Plot 2: Silhouette Score
ax2.plot(k_range, silhouette_scores, marker='s', linestyle='-', color='g')
ax2.set_title('Silhouette Score (Cari Titik Tertinggi)')
ax2.set_xlabel('Jumlah Klaster (k)')
ax2.set_ylabel('Score (Semakin mendekati 1 semakin baik)')
ax2.set_xticks(k_range)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Mencetak skor tertingginya agar mudah dibaca
best_k_sil = k_range[silhouette_scores.index(max(silhouette_scores))]
print(f"Berdasarkan Silhouette Score, K terbaik adalah: {best_k_sil} dengan skor {max(silhouette_scores):.4f}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Berdasarkan hasil evaluasi visual (Silhouette Score tertinggi), K optimal adalah 2
optimal_k = 2

print(f"Melatih model K-Means final dengan n_clusters = {optimal_k}...")

# 1. Inisialisasi model K-Means Final
kmeans_final = KMeans(
    n_clusters=optimal_k, 
    random_state=RANDOM_SEED, # Wajib: mengikuti informasi dari Orang 1
    n_init="auto"
)

kmeans_labels = kmeans_final.fit_predict(X_scaled)

# MASALAH SEBELUMNYA: Kamu harus memasukkan label ini ke df_model agar bisa dipanggil
df_model['KMeans_Cluster'] = kmeans_labels

print("K-Means berhasil dijalankan pada data yang telah dibersihkan!")

# 2. Latih model dan dapatkan label final untuk 361 perusahaan
kmeans_labels_final = kmeans_final.fit_predict(X_scaled)

# 3. Evaluasi metrik untuk dilaporkan
final_sil_score = silhouette_score(X_scaled, kmeans_labels_final)
final_dbi_score = davies_bouldin_score(X_scaled, kmeans_labels_final)

print("\n=== HASIL FINAL K-MEANS ===")
print(f"Jumlah Klaster (K)   : {optimal_k}")
print(f"Silhouette Score     : {final_sil_score:.4f} (Mendekati 1 = Sangat Baik)")
print(f"Davies-Bouldin Index : {final_dbi_score:.4f} (Mendekati 0 = Sangat Baik)")

# 4. Integrasi Output: Simpan hasil ke dataframe milik Orang 1
df_pca['KMeans_Cluster'] = kmeans_labels_final

# 5. Cek distribusi anggotanya (Berapa perusahaan di klaster 0, berapa di klaster 1)
print("\nDistribusi jumlah perusahaan per klaster:")
print(df_pca['KMeans_Cluster'].value_counts())

print("\nContoh 10 data pertama dengan label final:")
print(df_pca[['cleaned_company', 'KMeans_Cluster']].head(10))

#### DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import numpy as np

# 1. Tetapkan min_samples
min_samples = 5

# 2. Hitung jarak setiap perusahaan ke 5 tetangga terdekatnya
nn = NearestNeighbors(n_neighbors=min_samples)
nn_fit = nn.fit(X_scaled)
distances, indices = nn_fit.kneighbors(X_scaled)

# 3. Urutkan jarak dari yang paling rapat ke yang paling renggang
distances = np.sort(distances[:, min_samples-1], axis=0)

# 4. GAMBAR GRAFIKNYA (Ini yang memunculkan visualisasi K-Distance)
plt.figure(figsize=(8, 5))
plt.plot(distances, color='purple', linewidth=2)
plt.title('K-Distance Graph untuk Tuning EPS')
plt.xlabel('Data Points (Diurutkan)')
plt.ylabel('Jarak (eps)')
plt.grid(True, linestyle='--', alpha=0.7)

# Beri garis bantu untuk menunjukkan di mana titik siku (eps = 1.5) berada
plt.axhline(y=1.5, color='r', linestyle=':', label='Titik Siku (eps=1.5)')
plt.legend()

plt.show()

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.xlabel('Data Points')
plt.ylabel('Jarak ke tetangga ke-5')
plt.title('K-Distance Plot')
plt.grid(True)
plt.show()

# Cari juga range eps yang menghasilkan noise reasonable
print("Statistik jarak:")
print(f"Min  : {distances.min():.3f}")
print(f"Mean : {distances.mean():.3f}")
print(f"Max  : {distances.max():.3f}")
print(f"P25  : {np.percentile(distances, 25):.3f}")
print(f"P75  : {np.percentile(distances, 75):.3f}")

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import pandas as pd

# 1. Menetapkan Hyperparameter berdasarkan hasil eksperimen sebelumnya
optimal_eps = 1.4
min_samples = 5

print(f"Melatih model DBSCAN final dengan eps = {optimal_eps} dan min_samples = {min_samples}...")

# 2. Inisialisasi dan Latih Model DBSCAN
dbscan_final = DBSCAN(eps=optimal_eps, min_samples=min_samples)
dbscan_labels_final = dbscan_final.fit_predict(X_scaled)

# 3. Integrasi Output: Simpan hasil ke SEMUA dataframe penting
# Tambahkan ke df_pca untuk memudahkan Orang 4 (Visualizer) membuat scatter plot
df_pca['DBSCAN_Cluster'] = dbscan_labels_final

# TAMBAHKAN INI: Simpan juga ke df utama (bahan baku untuk Orang 3 dikirim ke ClickHouse)
df['DBSCAN_Cluster'] = dbscan_labels_final

# 4. Memisahkan data valid (klaster) dan data Noise (-1) untuk evaluasi
mask_no_noise = dbscan_labels_final != -1
labels_no_noise = dbscan_labels_final[mask_no_noise]
X_no_noise = X_scaled[mask_no_noise]

# Cek berapa banyak klaster valid yang terbentuk (tanpa noise)
n_valid_clusters = len(set(labels_no_noise))

print("\n=== HASIL FINAL DBSCAN ===")
if n_valid_clusters > 1:
    final_sil_dbscan = silhouette_score(X_no_noise, labels_no_noise)
    final_dbi_dbscan = davies_bouldin_score(X_no_noise, labels_no_noise)
    print(f"Jumlah Klaster Valid : {n_valid_clusters}")
    print(f"Silhouette Score     : {final_sil_dbscan:.4f} (Tanpa Noise)")
    print(f"Davies-Bouldin Index : {final_dbi_dbscan:.4f}")
elif n_valid_clusters == 1:
    print(f"Jumlah Klaster Valid : {n_valid_clusters} (Hanya terbentuk 1 klaster raksasa)")
    print("Metrik Kualitas      : Tidak bisa dihitung karena tidak ada klaster pembanding.")
else:
    print("Semua data dianggap noise!")

# 5. Cek Distribusi untuk analisis Bab 5
print("\nDistribusi jumlah perusahaan per klaster (DBSCAN):")
print(df['DBSCAN_Cluster'].value_counts().sort_index())

print("\nContoh 10 data pertama dengan label final:")
print(df_pca[['cleaned_company', 'DBSCAN_Cluster']].head(10))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Membuat Tabel Perbandingan Kuantitatif
comparison_data = {
    'Metrik Evaluasi': ['Silhouette Score', 'Davies-Bouldin Index', 'Jumlah Klaster'],
    'K-Means (K=2)': [final_sil_score, final_dbi_score, 2],
    'DBSCAN (eps=1.4)': [final_sil_dbscan, final_dbi_dbscan, 2]
}

df_comparison = pd.DataFrame(comparison_data)
print("=== TABEL PERBANDINGAN MODEL ===")
print(df_comparison)

# 2. Membuat Grafik Perbandingan (Untuk File 3 Visualisasi - Orang 4)
# Kita bandingkan Silhouette Score-nya
models = ['K-Means', 'DBSCAN']
scores = [final_sil_score, final_sil_dbscan]

plt.figure(figsize=(8, 5))
sns.barplot(x=models, y=scores, palette='viridis')
plt.title('Perbandingan Kualitas Klaster (Silhouette Score)', fontsize=14)
plt.ylabel('Silhouette Score (Makin Tinggi Makin Baik)')
plt.ylim(0, 1) # Range Silhouette adalah -1 sampai 1, tapi kita fokus di area positif
sns.barplot(x=models, y=scores, hue=models, palette='viridis', legend=False)

# Simpan untuk Orang 4
plt.savefig('05_perbandingan_silhouette.png', dpi=150)
plt.show()

In [ ]:
# Sekarang kolom 'KMeans_Cluster' sudah ada di df_model
print("Distribusi KMeans")
print(df_model['KMeans_Cluster'].value_counts())

# Cek distribusi DBSCAN
print("\nDistribusi DBSCAN:")
print(df['DBSCAN_Cluster'].value_counts())